# 05 — Prescriptive: from a score to a decision

Mirroring `references/Day5_Team_Workbook.ipynb`: *a prediction is not a decision.*

**Read this first.** The data is synthetic. Every cost and value below is a unitless
weight chosen to demonstrate the method, not a measured quantity. Nothing here is a
recommendation about anything. What this notebook shows is the arithmetic that turns a
ranking into a decision — and, applied honestly, that arithmetic says do not act.


Reusable logic belongs in `src/startup_outcomes/`, not here — this notebook imports it and
defines nothing. Strip all outputs before committing, and check that it still runs under
**Restart & Run All**.

In [ ]:
# Activates the inline backend so a returned Figure renders. plots.py itself never
# imports pyplot, so no global state is created by the library.
%matplotlib inline

from startup_outcomes import features, plots, report
from startup_outcomes.load import load_raw
from startup_outcomes.models import prescriptive, protocol, supervised

df = load_raw(verify_checksum=True)
analysis = features.canonical_frame(df)
design, target = features.design_matrix(analysis)
scores = protocol.out_of_fold_probabilities(supervised.chosen_estimator(), design, target)

Those scores are out-of-fold: no company is ranked by a model that trained on it. The
reference workbook fits on 75% of rows and then scores all of them, which inflates every
lift, threshold and expected value it reports.

## Step 1 — what-if: how the prediction moves

Swept across the realistic range of one feature, averaged over all rows.

In [ ]:
curve = prescriptive.response_curve(design, target, "Runway_Months_2024")
print(prescriptive.response_summary(design, target, "Runway_Months_2024"))
plots.response_curve(curve, base_rate_pct=float(target.mean() * 100))

This shows how the model's prediction changes, **not** what would happen in the real world
if a company's runway changed. Every serious analyst is eventually asked to forget that
distinction. Don't.

## Step 2 — the ranked action list

The top of a ranking produced by a model that does not beat a single column, over
companies that do not exist. Note that every row is short of runway: the model is
restating the rule.

In [ ]:
print(prescriptive.ranked_actions(analysis, scores, top=15).to_string())
print()
print(prescriptive.targeting_mix(analysis, scores, "Domain").head(8).to_string())

## Step 3 — the cost of being wrong

Two editable weights. Change `cost_miss` to 3 and re-run, then to 50: the cheapest
threshold moves, and nothing about the model changed. That is prescriptive analytics in
one cell — the threshold is a business decision, not a technical one.

In [ ]:
costs = prescriptive.cost_threshold_sweep(target, scores, cost_miss=10.0)
print(costs.to_string())
verdict = prescriptive.cheapest_threshold(target, scores, cost_miss=10.0)
for key, value in verdict.items():
    print(f"  {key:<38} {value}")

In [ ]:
plots.threshold_cost_curve(
    costs,
    cheapest_threshold=float(verdict["cheapest_threshold"]),
    flag_everyone_cost=float(verdict["cost_flag_everyone_units"]),
)

Read the comparator, not the optimum. The cost-minimising policy flags 97% of companies
and saves 1.7% against simply flagging everyone. The model is steering almost nothing.

## Step 4 — is the intervention worth doing at all?

The hit rate below comes from the true labels of the targeted companies, and the model's
own average score is reported separately. The gap between them is a calibration finding
the workbooks' formulation would have hidden.

In [ ]:
value = prescriptive.capacity_expected_value(target, scores, capacity=30)
for key, item in value.items():
    print(f"  {key:<38} {item}")

Six events. The interval on the realised rate spans the base rate, so targeting the top
thirty is **indistinguishable from picking at random**, and the apparent 1.42× lift is not
resolvable at this capacity.

## Step 5 — sensitivity: how wrong can you afford to be?

Find the row where the sign flips. That is the margin of safety.

In [ ]:
sensitivity = prescriptive.value_sensitivity(target, scores)
print(sensitivity.to_string())
plots.value_sensitivity(sensitivity)

## Step 6 — the three recommendations

1. **Action:** do not deploy this model as a targeting tool.
   **Applies to:** any use that ranks companies for intervention.
   **Expected effect:** none foregone — targeting the top 30 is indistinguishable from
   random at this capacity.
   **How we would know:** the interval on the realised rate stops containing the base rate.
   **What would make this wrong:** a capacity large enough for the top-decile lift of 1.68
   to be resolvable.

2. **Action:** if a screen is needed, use the runway threshold directly.
   **Applies to:** the same use case, at a fraction of the complexity.
   **Expected effect:** PR-AUC 0.191 against the model's 0.193, a difference smaller than
   the fold spread.
   **How we would know:** a model clears the runway column by more than one fold SD.
   **What would make this wrong:** a feature arriving that is not a size proxy.

3. **Action:** treat the 6-month threshold as a generator artifact, not a finding.
   **Applies to:** any attempt to read these results as being about real startups.
   **Expected effect:** prevents a synthetic rule being repeated as a fact.
   **How we would know:** the dataset author confirms the generating process.
   **What would make this wrong:** the data turning out to be real, which the audit's seven
   internal contradictions rule out.

## Step 7 — capstone

In [ ]:
print(report.summary_text())

## Final checklist

- Every number above has an `n` or an interval beside it.
- The word *causes* appears nowhere. No experiment was run.
- The "how much to believe it" step is not optional: the fold spread, the intervals and the
  seed-instability range are all reported, and they are what make the headline negative.

## Scratch